In [ ]:
import os
import csv
import yaml
import pandas as pd

In [ ]:
semgrep_rules_root = r"<SEMGREP_REPO_RULESET>"
csv_pfad = "all_semgrep_findings.csv"
df = pd.read_csv(csv_pfad)

In [ ]:
rule_names = df["applied_semgrep_rule"].dropna().unique()
rule_to_csv_cwe = df.groupby("applied_semgrep_rule")["found_cwe_id"].apply(lambda x: x.dropna().unique())

In [ ]:
data = []

for rule_name in sorted(rule_names):
    parts = rule_name.split(".")
    if len(parts) < 2:
        print(f"rule name not valid {rule_name}")
        continue

    yaml_filename = parts[-2] + ".yaml"
    yaml_pfad = os.path.join(semgrep_rules_root, *parts[:-2], yaml_filename)

    entry = {
        "rule_name": rule_name,
        "cwe_id": None,
        "rule_found": False,
        "matches_csv": False
    }

    cwe_ids_csv = rule_to_csv_cwe.get(rule_name, [])

    if not os.path.isfile(yaml_pfad):
        print(f"couldn't find yaml for {rule_name}")
        entry["rule_found"] = False
        if len(cwe_ids_csv) > 0:
            entry["rule_found"] = True
            entry["matches_csv"] = True
            entry["cwe_id"] = cwe_ids_csv[0] 
        data.append(entry)
        continue

    entry["rule_found"] = True

    try:
        with open(yaml_path, "r", encoding="utf-8") as f:
            yaml_content = yaml.safe_load(f)
    except Exception as e:
        print(f"error loading {yaml_path}: {e}")
        data.append(entry)
        continue

    if not isinstance(yaml_content, dict) or "rules" not in yaml_content:
        print(f"yaml structure invalid {yaml_path}")
        data.append(entry)
        continue

    for rule in yaml_content.get("rules", []):
        metadata = rule.get("metadata", {})
        cwes = metadata.get("cwe", [])

        if not cwes:
            continue

        for cwe in cwes:
            if isinstance(cwe, str) and ":" in cwe:
                cwe_id_raw, cwe_desc = cwe.split(":", 1)
                yaml_cwe_id = cwe_id_raw.strip().lower().replace("cwe-", "cwe-")
                entry["cwe_id"] = yaml_cwe_id

                if yaml_cwe_id in [c.strip().lower() for c in cwe_ids_csv]:
                    entry["matches_csv"] = True
                break
        break  

    if not entry["cwe_id"] and len(cwe_ids_csv) > 0:
        entry["cwe_id"] = cwe_ids_csv[0]
        entr["matches_csv"] = True

    data.append(entry)
    

In [ ]:
df_results = pd.DataFrame(data)
df_results = df_results.drop_duplicates(subset=["rule_name"])
df_results.head()

In [ ]:
rules_wo_cwe = df_results[df_results["cwe_id"].isnull() | (df_resulsts["cwe_id"].str.strip() == "")]

print(f"rules without assigned cwe-id: {len(rules_wo_cwe)}")

In [ ]:
rules_wo_cwe_list = rules_wo_cwe["rule_name"].tolist()

invalid_findings_count = df[df["applied_semgrep_rule"].isin(rules_wo_cwe_list)].shape[0]
print(f"findings with rules without assigned cwe-id {invalid_findings_count}")

df["sample_name"] = (df["file_path"].str.replace("code_before/", "", regex=False).str.replace("/", "_", regex=False).str.lower())
df["groundtruth_cwe_id"] = df["groundtruth_cwe_id"].str.replace(r"cwe-0(\d\d)", r"cwe-\1", regex=True)

df_filtered = df[~df["applied_semgrep_rule"].isin(rules_wo_cwe_list)]
print(f"resulting findings after filtering {df_gefiltert.shape[0]}")


In [ ]:
df_filtered.head()

In [ ]:
df_wo_cwe = df[df["applied_semgrep_rule"].isin(rules_wo_cwe_list)]

unique_samples_wo_cwe = df_wo_cwe["sample_name"].unique()
print(f"samples with rules without assigned cwe: {len(unique_samples_wo_cwe)}")

samples_with_other_findings = df_filtered[df_filtered["sample_name"].isin(unique_samples_wo_cwe)]["sample_name"].unique()
samples_no_other_findings = set(unique_samples_wo_cwe) - set(samples_with_other_findings)

print(f"samples filtered out due to no assigned cwe in any findings {len(samples_no_other_findings)}")
print(f"samples with other valid findings {len(samples_with_other_findings)}")

In [ ]:
df_filtered.to_csv("rulemapped_and_cleaned_findings.csv", index=False)

In [ ]:
df_filtered.info()